# 07 — Generate marketing campaigns

**Business objective:** we want a plausible *root cause*, not just a symptom.
Reduced marketing spend targeted at Sydney during the anomaly window is the
causal story this MVP tells: less marketing → fewer orders → lower revenue.

**What we're generating:** ~30 campaigns across cities and channels, with
Sydney-targeted spend reduced during the anomaly window.

In [1]:
import sys
sys.path.insert(0, '..')
from notebooks import nb_config as cfg

import pandas as pd
import numpy as np
from faker import Faker

rng = np.random.default_rng(cfg.SEED + 6)
fake = Faker()
Faker.seed(cfg.SEED + 6)

stores = pd.read_csv(cfg.DATA_DIR / "stores.csv")
cities = sorted(stores["city"].unique())
channels = ["Social", "Search", "Email", "Display"]

## Generation logic

Campaigns run in overlapping windows across the 18-month history. Any campaign targeting Sydney with a start date inside the anomaly window gets a reduced budget — this is the causal lever behind the order-volume drop in notebook 04.

In [2]:
end_date = pd.Timestamp("2026-07-01")
start_date = end_date - pd.DateOffset(months=cfg.ORDER_HISTORY_MONTHS)
anomaly_start = end_date - pd.DateOffset(months=cfg.ANOMALY_MONTHS)
window_days = (end_date - start_date).days

n_campaigns = 30
rows = []
for i in range(1, n_campaigns + 1):
    city = rng.choice(cities)
    c_start = start_date + pd.Timedelta(days=int(rng.uniform(0, window_days - 30)))
    c_end = c_start + pd.Timedelta(days=int(rng.uniform(14, 45)))
    budget = float(rng.uniform(5_000, 40_000))
    if city == cfg.ANOMALY_CITY and c_start >= anomaly_start:
        budget *= (1 - cfg.ANOMALY_ORDER_REDUCTION)  # same lever as the order-volume cut
    rows.append({
        "campaign_id": i,
        "campaign_name": fake.catch_phrase(),
        "city_target": city,
        "channel": rng.choice(channels),
        "start_date": c_start,
        "end_date": c_end,
        "budget": round(budget, 2),
        "spend": round(budget * rng.uniform(0.85, 1.0), 2),
    })

campaigns = pd.DataFrame(rows)
campaigns.head()

,campaign_id,campaign_name,city_target,channel,start_date,end_date,budget,spend
0,1,Reactive foreground capacity,Adelaide,Search,2025-11-04,2025-12-03,29307.22,27779.56
1,2,Realigned transitional monitoring,Sydney,Display,2025-11-01,2025-12-01,32010.21,28219.60
2,3,Innovative motivating core,Perth,Social,2025-02-14,2025-03-14,18187.35,15949.34
3,4,Universal well-modulated Internet solution,Perth,Search,2025-12-12,2026-01-17,30899.14,27677.52
4,5,Down-sized coherent workforce,Canberra,Social,2026-01-29,2026-03-04,21246.95,19273.95


## Sample output & distribution check

In [3]:
print(campaigns.groupby("city_target")["spend"].agg(["count", "sum"]))

             count        sum
city_target                  
Adelaide         6  128958.54
Brisbane         4   39479.32
Canberra         5   63070.13
Melbourne        8  197088.99
Perth            4   82769.69
Sydney           3   66200.69


## Validation

In [4]:
assert campaigns["campaign_id"].is_unique
assert (campaigns["end_date"] > campaigns["start_date"]).all()
assert campaigns["city_target"].isin(cities).all()
assert campaigns.isnull().sum().sum() == 0
print("All validation checks passed.")

All validation checks passed.


In [5]:
out_path = cfg.DATA_DIR / "marketing_campaigns.csv"
campaigns.to_csv(out_path, index=False)
print(f"Wrote {len(campaigns):,} rows to {out_path}")

Wrote 30 rows to /home/claude/project/eaida/data/raw/marketing_campaigns.csv


## Summary

Generated 30 campaigns across cities and channels, with Sydney campaigns
starting in the anomaly window getting reduced budgets — the intended root
cause behind the order-volume and revenue drop. Saved to
`data/raw/marketing_campaigns.csv`.